# Schema Enforcement and Evolution
>Delta Lake enforces schema by default. Not as an option you configure — it is always on.

In [0]:
USE CATALOG workspace;
USE SCHEMA default;

####1. Create the products target table

In [0]:
CREATE OR REPLACE TABLE products (
  product_id  BIGINT     COMMENT 'Unique product identifier',
  name        STRING  COMMENT 'Product name',
  category    STRING  COMMENT 'Product category',
  price       DOUBLE  COMMENT 'Unit price in USD'
)
USING DELTA
COMMENT 'Product catalogue — used for schema evolution demos';

####2. Load initial data

In [0]:
INSERT INTO products VALUES
  (1, 'Wireless Headphones', 'Electronics', 79.99),
  (2, 'Standing Desk',       'Furniture',   349.00),
  (3, 'USB-C Hub',           'Electronics', 44.99),
  (4, 'Monitor Stand',       'Furniture',   55.00);

####3. BAD WRITE 1 — extra column in source

In [0]:
%python
from pyspark.sql import Row

bad_data_1 = [
    Row(product_id=5, name='Mechanical Keyboard',
        category='Electronics', price=129.50, discount_pct=0.10),
    Row(product_id=6, name='Desk Lamp',
        category='Furniture',   price=34.99,  discount_pct=0.05),
]

df_bad_1 = spark.createDataFrame(bad_data_1)

# This write WILL FAIL — discount_pct is not in the table schema
df_bad_1.write \
    .format('delta') \
    .mode('append') \
    .saveAsTable('products')

>if your incoming data has columns the table does not know about, \
Delta rejects the write and asks you to be explicit about what you want to do.

####4. BAD WRITE 2: Type mismatch

In [0]:
%python
bad_data_2 = [
    Row(product_id=5, name='Mechanical Keyboard',
        category='Electronics', price='129.50'),   # <-- STRING, not DOUBLE
]

df_bad_2 = spark.createDataFrame(bad_data_2)

# This write WILL FAIL — incompatible type for the price column
df_bad_2.write \
    .format('delta') \
    .mode('append') \
    .saveAsTable('products')

### Two approaches — mergeSchema vs autoMerge


####1. Sub-demo 1 — mergeSchema adds discount_pct

In [0]:
%python
new_products = [
    Row(product_id=5, name='Mechanical Keyboard',
        category='Electronics', price=129.50, discount_pct=0.10),
    Row(product_id=6, name='Desk Lamp',
        category='Furniture',   price=34.99,  discount_pct=0.05),
]

df_new = spark.createDataFrame(new_products)

# mergeSchema allows Delta to add the new column
df_new.write \
    .format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('products')

print('Write with mergeSchema succeeded.')

####2.  Verify — new column added

In [0]:
DESCRIBE TABLE products;

####3. see NULL-filling for existing rows

In [0]:
SELECT product_id, name, price, discount_pct
FROM   products
ORDER  BY product_id;

####4.  Sub-demo 2 — the rename gotcha
>Writing discount_rate instead of discount_pct — expecting a rename\
but mergeSchema will ADD a new column, not rename the existing one

In [0]:
%python
renamed_products = [
    Row(product_id=7, name='Ergonomic Mouse',
        category='Electronics', price=59.99, discount_rate=0.08),  # renamed column
]

df_renamed = spark.createDataFrame(renamed_products)

df_renamed.write \
    .format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('products')

print('Write succeeded. Now check the schema — you may be surprised.')

>The rename gotcha — two columns instead of one rename\
mergeSchema cannot rename — use ALTER TABLE RENAME COLUMN instead

In [0]:
%sql
select * from products;

In [0]:
DESCRIBE TABLE products;

### ALTER TABLE and Generated Columns

####1. ALTER TABLE ADD COLUMN — metadata only, instant

In [0]:
ALTER TABLE products
ADD COLUMN supplier STRING COMMENT 'Product supplier name';

>it adds the column to the schema stored in the transaction log without touching any Parquet files.\
This is why it completes in milliseconds on a billion-row table.

In [0]:
DESCRIBE HISTORY products;

####2. ALTER TABLE RENAME COLUMN

>Enable Column Mapping on your Delta table if you need to rename a column.\
It may be enabled by default in futue versions.

In [0]:
ALTER TABLE products
SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion'   = '2',
  'delta.minWriterVersion'   = '5'
);

>The correct way to rename\
This is metadata-only — no Parquet files are rewritten


In [0]:
ALTER TABLE products
RENAME COLUMN discount_rate TO discount_rate_v2;

####3. ALTER TABLE DROP COLUMN

In [0]:
ALTER TABLE products DROP COLUMN supplier;

####3. Generated columns

In [0]:
CREATE OR REPLACE TABLE products_v2 (
  product_id     BIGINT,
  name           STRING,
  price          DOUBLE,
  price_with_tax DOUBLE GENERATED ALWAYS AS (ROUND(price * 1.20, 2))
)
USING DELTA
COMMENT 'Products with a tax-inclusive price computed automatically';

>insert without specifying price_with_tax - Delta computes it

In [0]:
INSERT INTO products_v2 (product_id, name, price) VALUES
  (1, 'Wireless Headphones', 79.99),
  (2, 'Standing Desk',       349.00),
  (3, 'USB-C Hub',           44.99);

>Verify — price_with_tax auto-populated from price

In [0]:
SELECT product_id, name, price, price_with_tax
FROM   products_v2
ORDER  BY product_id;

####Summary
Four things to take away from this lecture.

Schema enforcement is your default safety net. It rejects writes with extra columns or type mismatches — before any damage reaches the table. 

To add new columns during a write — use mergeSchema. Per-write, intentional, production-safe.\
Existing rows get NULL in the new column.\
mergeSchema cannot rename — only add. For true renames, use ALTER TABLE RENAME COLUMN.

ALTER TABLE operations are metadata-only. ADD, RENAME, and DROP columns update the schema in the transaction log without rewriting any Parquet files. Instant on any table size.\
DROP removes the column from queries immediately but does not erase the physical data — OPTIMIZE and VACUUM complete the erasure.

Generated columns are computed at write time and stored physically. Define them with GENERATED ALWAYS AS (expression).\
Delta populates them automatically on every INSERT or UPDATE. Remember, they are physically stored.